<h1 id="setup-and-configuration"
    style="
  background: linear-gradient(135deg, #0f2027, #203a43, #2c5364);
  color: white;
  padding: 12px 24px;
  text-align: center;
  border-radius: 12px;
  font-size: 22px;
  font-weight: bold;
  font-family: Arial, sans-serif;
  margin: 20px 0;
">
  1. Imports and Configurations
</h1>

In [1]:
import sys
import random
import pickle
import yaml

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from torch.utils.data import Dataset, DataLoader
import plotly.graph_objects as go

class Config:
    sample_sub = "/kaggle/input/stanford-rna-3d-folding-2/sample_submission.csv"
    test_seq = "/kaggle/input/stanford-rna-3d-folding-2/test_sequences.csv"
    train_labels = "/kaggle/input/stanford-rna-3d-folding-2/train_labels.csv"
    train_sequences = "/kaggle/input/stanford-rna-3d-folding-2/train_sequences.csv"
    validation_labels = "/kaggle/input/stanford-rna-3d-folding-2/validation_labels.csv"
    validation_sequences = "/kaggle/input/stanford-rna-3d-folding-2/validation_sequences.csv"
    model_config_path = "/kaggle/input/ribonanzanet2d-final/configs/pairwise.yaml"
    pretrained_weights_path = "/kaggle/input/ribonanzanet-weights/RibonanzaNet.pt"
    finetuned_weights_path = "/kaggle/input/ribonanzanet-finetuned/RibonanzaNet-3D.pt"
    
    max_len = 384
    batch_size = 1
    seed = 42
    num_tta = 5  

config = Config()

# Set seed for reproducibility
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(config.seed)

#  Load Test Data
test_data = pd.read_csv(config.test_seq)
print(f"Test sequences: {len(test_data)}")

Test sequences: 28


<h1 id="loader-set"
    style="
  background: linear-gradient(135deg, #0f2027, #203a43, #2c5364);
  color: white;
  padding: 12px 24px;
  text-align: center;
  border-radius: 12px;
  font-size: 22px;
  font-weight: bold;
  font-family: Arial, sans-serif;
  margin: 20px 0;
">
  2. Test Dataset
</h1>

In [2]:
class RNADataset(Dataset):
    def __init__(self, data, max_len=384):
        self.data = data
        self.max_len = max_len
        self.tokens = {nt: i for i, nt in enumerate("ACGU")}

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sequence = [self.tokens[nt] for nt in self.data.loc[idx, "sequence"]]
        sequence = torch.tensor(np.array(sequence), dtype=torch.long)
        
        original_length = len(sequence)
        
        return {
            "sequence": sequence,
            "original_length": original_length,
            "target_id": self.data.loc[idx, "target_id"]
        }


test_dataset = RNADataset(test_data, max_len=config.max_len)

<h1 id="model-diff"
    style="
  background: linear-gradient(135deg, #0f2027, #203a43, #2c5364);
  color: white;
  padding: 12px 24px;
  text-align: center;
  border-radius: 12px;
  font-size: 22px;
  font-weight: bold;
  font-family: Arial, sans-serif;
  margin: 20px 0;
">
  3.  Model Definition
</h1>

In [3]:
sys.path.append("/kaggle/input/ribonanzanet2d-final")
from Network import RibonanzaNet


class Config:
    def __init__(self, **entries):
        self.__dict__.update(entries)
        self.entries = entries

    def print(self):
        print(self.entries)


def load_config_from_yaml(file_path):
    with open(file_path, "r") as file:
        cfg = yaml.safe_load(file)
    return Config(**cfg)


class FinetunedRibonanzaNet(RibonanzaNet):
    def __init__(self, config_obj, pretrained=False, dropout=0.1):
        config_obj.dropout = dropout
        super(FinetunedRibonanzaNet, self).__init__(config_obj)

        if pretrained:
            self.load_state_dict(
                torch.load(config.pretrained_weights_path, map_location="cpu")
            )

        self.dropout = nn.Dropout(p=0.0)
        self.xyz_predictor = nn.Linear(256, 3)

    def forward(self, src):
        sequence_features, _ = self.get_embeddings(
            src, torch.ones_like(src).long().to(src.device)
        )
        xyz_pred = self.xyz_predictor(sequence_features)
        return xyz_pred


#  Load Model
model_cfg = load_config_from_yaml(config.model_config_path)
model = FinetunedRibonanzaNet(model_cfg, pretrained=False, dropout=0.2).cuda()

# Load finetuned weights
model.load_state_dict(
    torch.load(config.finetuned_weights_path, map_location="cuda")
)
print("Model loaded successfully!")



constructing 9 ConvTransformerEncoderLayers
Model loaded successfully!


<h1 id="inference"
    style="
  background: linear-gradient(135deg, #0f2027, #203a43, #2c5364);
  color: white;
  padding: 12px 24px;
  text-align: center;
  border-radius: 12px;
  font-size: 22px;
  font-weight: bold;
  font-family: Arial, sans-serif;
  margin: 20px 0;
">
  4.  Inference
</h1>

In [4]:
def predict_sequence(model, sequence, max_len=384):
    """
    Predict XYZ coordinates for a sequence.
    Handles sequences longer than max_len by sliding window approach.
    """
    device = next(model.parameters()).device
    seq_len = len(sequence)
    
    if seq_len <= max_len:
        # Sequence fits in one pass
        src = sequence.unsqueeze(0).to(device)
        
        model.eval()
        with torch.no_grad():
            xyz = model(src).squeeze(0)
        
        return xyz.cpu().numpy()
    
    else:
        # Sequence is longer than max_len - use sliding window
        step_size = max_len // 2  # 50% overlap
        predictions_sum = np.zeros((seq_len, 3))
        counts = np.zeros(seq_len)
        
        for start in range(0, seq_len - max_len + 1, step_size):
            end = start + max_len
            window = sequence[start:end].unsqueeze(0).to(device)
            
            model.eval()
            with torch.no_grad():
                xyz = model(window).squeeze(0)
            
            window_pred = xyz.cpu().numpy()
            predictions_sum[start:end] += window_pred
            counts[start:end] += 1
        
        # Handle the last segment if it doesn't align
        if (seq_len - max_len) % step_size != 0:
            start = seq_len - max_len
            window = sequence[start:].unsqueeze(0).to(device)
            
            model.eval()
            with torch.no_grad():
                xyz = model(window).squeeze(0)
            
            window_pred = xyz.cpu().numpy()
            predictions_sum[start:] += window_pred
            counts[start:] += 1
        
        # Average overlapping predictions
        final_pred = predictions_sum / counts[:, np.newaxis]
        return final_pred



# Run Inference

print("Running inference...")
all_predictions = []

for i in range(len(test_dataset)):
    sample = test_dataset[i]
    sequence = sample["sequence"]
    
    # Predict
    pred = predict_sequence(model, sequence, max_len=config.max_len)
    all_predictions.append(pred)
    
    if (i + 1) % 10 == 0:
        print(f"Processed {i + 1}/{len(test_dataset)} sequences")

print("Inference complete!")


Running inference...


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


Processed 10/28 sequences
Processed 20/28 sequences
Inference complete!


<h1 id="submission"
    style="
  background: linear-gradient(135deg, #0f2027, #203a43, #2c5364);
  color: white;
  padding: 12px 24px;
  text-align: center;
  border-radius: 12px;
  font-size: 22px;
  font-weight: bold;
  font-family: Arial, sans-serif;
  margin: 20px 0;
">
  5.  Submission
</h1>

In [5]:
data = []

for i in range(len(test_data)):
    target_id = test_data.loc[i, "target_id"]
    sequence = test_data.loc[i, "sequence"]
    seq_length = len(sequence)
    
    # Get predictions for this sequence
    preds = all_predictions[i]  
    
    for j in range(seq_length):
        row = [
            f"{target_id}_{j+1}",
            sequence[j],
            j + 1,
        ]
        
        # Add the same prediction 5 times 
        for k in range(5):
            row.extend([preds[j, 0], preds[j, 1], preds[j, 2]])
        
        data.append(row)

# Create submission DataFrame
columns = ["ID", "resname", "resid"]
for i in range(1, 6):
    columns += [f"x_{i}", f"y_{i}", f"z_{i}"]

submission = pd.DataFrame(data, columns=columns)

# Verify submission format
print(f"\nSubmission shape: {submission.shape}")
print(submission.head())

# Save submission
submission.to_csv("submission.csv", index=False)
print("\nSubmission saved to submission.csv")


Submission shape: (9762, 18)
       ID resname  resid       x_1        y_1        z_1       x_2        y_2  \
0  8ZNQ_1       A      1  0.672380  16.071049  20.171282  0.672380  16.071049   
1  8ZNQ_2       C      2  2.558037  11.287163  19.240793  2.558037  11.287163   
2  8ZNQ_3       C      3  1.933371   6.633279  16.606203  1.933371   6.633279   
3  8ZNQ_4       G      4 -0.237766   2.789619  12.873806 -0.237766   2.789619   
4  8ZNQ_5       U      5 -3.124321   1.368961   8.007702 -3.124321   1.368961   

         z_2       x_3        y_3        z_3       x_4        y_4        z_4  \
0  20.171282  0.672380  16.071049  20.171282  0.672380  16.071049  20.171282   
1  19.240793  2.558037  11.287163  19.240793  2.558037  11.287163  19.240793   
2  16.606203  1.933371   6.633279  16.606203  1.933371   6.633279  16.606203   
3  12.873806 -0.237766   2.789619  12.873806 -0.237766   2.789619  12.873806   
4   8.007702 -3.124321   1.368961   8.007702 -3.124321   1.368961   8.007702   

  